# COMPOSITE VOICE INSTABILITY INDEX (JITTER + SHIMMER)

## $ Jitter = \frac{1}{N-1} Σ_{i=1}^{N-1} |T_i - T_{i+1}| / (\frac{1}{N} Σ_{i=1}^{N} T_i) $

## $ Shimmer = \frac{1}{N-1} Σ_{i=1}^{N-1} |A_i - A_{i+1}| / (\frac{1}{N} Σ_{i=1}^{N} A_i) $

In [2]:
import numpy as np
import librosa
import matplotlib.pyplot as plt

In [31]:
def compute_jitter_shimmer(audio, sr, f0_min=70, f0_max=500):

    # pitch using pyin

    f0, voiced_flag, _= librosa.pyin(audio, fmin=f0_min, fmax=f0_max, sr=sr, frame_length= 2048, hop_length=512, resolution=0.1)

    # filter voiced frames

    f0_voiced = f0[voiced_flag]

    if len(f0_voiced)<2:
        return 0.0, 0.0, 0.0

    # periods

    periods = 1/f0_voiced

    # jitter
    if len(periods)>1:
        
        diff = np.abs(np.diff(periods))
        mean_diff = np.mean(diff)
        mean_periods = np.mean(periods)
        jitter = mean_diff/mean_periods
    else:
        jitter = 0.0


    # Now Shimmer : we need amplitudes

    envelope = np.abs(librosa.stft(audio))
    envelope_mean = np.mean(envelope, axis=0)

    frame_size = int(sr * 0.025) # 25ms frames

    hop_size = int(sr * 0.010) # 10ms hop

    rms = librosa.feature.rms(y=audio, frame_length=frame_size, hop_length=hop_size)[0]

    voiced_indices = np.where(voiced_flag)[0]
    if len(voiced_indices) > 1 and len(rms) > 0:
        rms_voiced = rms[:len(voiced_indices)]
        if len(rms_voiced) > 1:
            rms_diffs = np.abs(np.diff(rms_voiced))
            shimmer = np.mean(rms_diffs) / np.mean(rms_voiced)

        else:
            shimmer = 0.0

    else:
        shimmer = 0.0


    jitter_norm = float(min(jitter * 100, 1.0))  # Convert to percentage, cap at 1
    shimmer_norm = float(min(shimmer * 100, 1.0))
    composite_index = float((jitter_norm + shimmer_norm) / 2)
    
    return jitter, shimmer, composite_index

In [32]:
sr = 16000

file = 'Jane.mp3'

audio, sr = librosa.load(file, sr=sr, mono=True)

In [33]:
compute_jitter_shimmer(audio, sr=sr,f0_min=75, f0_max=300)

(np.float64(0.03604279552979921), np.float32(0.08676871), 1.0)

In [10]:
len(audio)/sr

2.0